## Deploy em Real Time

Fiz a disponibilização do endpoints em https://dbc-f0228801-51ae.cloud.databricks.com/serving-endpoints/modelo_propensao/invocations.
Para o meu modelo de machine learning que está no catalogo unity propensao_compra_modelo. Com isso faz um código que simule os dados de cliente chegando em tempo real e realize a previsão do modelo. Lembrando de manter as colunas do dataframe corretas. Um Exemplo abaixo.
            "idade_cliente": 34,
            "renda_mensal_k": 12.5,
            "tempo_medio_clique_segundos": 45.2,
            "media_interacoes_suporte": 1,
            "media_cupons_ativos": 2,
            "media_score_nps_cliente": 9.0,
            "media_dias_inatividade": 5,
            "total_gasto_acumulado_reais": 1500.0,
            "genero_cliente_m": 0,
            "regiao_cliente_nordeste": 0,
            "regiao_cliente_norte": 0,
            "regiao_cliente_sudeste": 1,
            "regiao_cliente_sul": 0

In [0]:
import random
import time
import pandas as pd
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()

# Função para gerar dados de cliente simulados em tempo real
def gerar_dados_cliente():
    regioes = ["nordeste", "norte", "sudeste", "sul"]
    regiao = random.choice(regioes)
    
    cliente = {
        "idade_cliente": random.randint(18, 75),
        "renda_mensal_k": round(random.uniform(1.0, 30.0), 2),
        "tempo_medio_clique_segundos": round(random.uniform(5.0, 120.0), 2),
        "media_interacoes_suporte": random.randint(0, 10),
        "media_cupons_ativos": random.randint(0, 5),
        "media_score_nps_cliente": round(random.uniform(0.0, 10.0), 1),
        "media_dias_inatividade": random.randint(0, 30),
        "total_gasto_acumulado_reais": round(random.uniform(0.0, 50000.0), 2),
        "genero_cliente_m": random.choice([0, 1]),
        "regiao_cliente_nordeste": 1 if regiao == "nordeste" else 0,
        "regiao_cliente_norte": 1 if regiao == "norte" else 0,
        "regiao_cliente_sudeste": 1 if regiao == "sudeste" else 0,
        "regiao_cliente_sul": 1 if regiao == "sul" else 0,
    }
    return cliente

# Simulação de chegada de dados em tempo real
print("=== Simulação de Previsões em Tempo Real ===\n")

resultados = []
for i in range(1, 6):
    dados_cliente = gerar_dados_cliente()
    
    print(f"Cliente {i} chegou:")
    print(f"  Idade: {dados_cliente['idade_cliente']} | Renda: R$ {dados_cliente['renda_mensal_k']}k")
    print(f"  Região: {next(k for k, v in dados_cliente.items() if k.startswith('regiao_cliente_') and v == 1).replace('regiao_cliente_', '').capitalize()}")
    print(f"  Gasto acumulado: R$ {dados_cliente['total_gasto_acumulado_reais']:.2f}")
    
    # Realiza a previsão via endpoint
    response = workspace.serving_endpoints.query(
        name="modelo_propensao",
        dataframe_records=[dados_cliente],
    )
    
    print(f"  Resposta do modelo: {response.predictions}")
    print("  " + "-" * 50)
    
    resultados.append({"cliente": i, **dados_cliente, "previsao": response.predictions})
    
    time.sleep(1)  # Simula intervalo entre chegadas

# Exibe os resultados consolidados
df_resultados = pd.DataFrame(resultados)
display(df_resultados)